# E015 — SD 1.5, SDXL et FLUX comme références esthétiques seulement

Cette expérience répond à une question limitée : **quel backbone produit la meilleure image de
référence pour nos quatre prompts ?** Elle ne compare ni ControlNet QR, ni SRPG, ni SR-MPGD, car
ceux-ci ne sont pas compatibles de façon identique avec les trois familles.

- SD 1.5 : Cetus-Mix Whalefall, utilisé par DiffQRCoder ;
- SDXL : `stabilityai/stable-diffusion-xl-base-1.0` ;
- FLUX : `black-forest-labs/FLUX.1-schnell`.

Après chaque référence, le même constructeur adaptatif exact-payload d'E014A est appliqué pour
mesurer son **potentiel d'intégration**, sans relancer une diffusion QR. Les modèles sont chargés
l'un après l'autre sur la RTX 4000 Ada 20 Go et entièrement libérés entre deux familles.


## Ce que le tableau final permettra — et ne permettra pas — de conclure

```text
même prompt + seed logique
       ├── SD1.5 / 30 pas / 768
       ├── SDXL  / 30 pas / 768
       └── FLUX  /  4 pas / 768
                │
                ├── temps, pic VRAM, CLIP-aes, CLIPScore
                └── exact mask + adaptatif → SSR du blueprint (pas d'image QR finale)
```

Le nombre de pas recommandé diffère par architecture : les temps représentent une recette
opérationnelle, pas un benchmark FLOP-à-FLOP. Une seed identique n'engendre pas le même bruit entre
architectures ; la comparaison est appariée par prompt, pas pixel-à-pixel.


In [ ]:
from __future__ import annotations

import gc
import json
import os
import shutil
import time
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentencepiece
import torch
from diffusers import FluxPipeline, StableDiffusionPipeline, StableDiffusionXLPipeline
from google.protobuf import __version__ as protobuf_version
from huggingface_hub import hf_hub_download, model_info
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError
from IPython.display import display
from PIL import Image

from prooftag_qr.blueprints import build_adaptive_blueprint, exact_mask_candidates
from prooftag_qr.geometry import AlignedQR
from prooftag_qr.quality_scoring import CLIPQualityScorer
from prooftag_qr.validation import QRValidator, summarize_validation_records

assert torch.cuda.is_available(), 'Lancer dans le pod GPU.'
print(torch.cuda.get_device_name(0))
print('Dépendances FLUX : sentencepiece', sentencepiece.__version__, '/ protobuf', protobuf_version)


## 1. Contrat, modèles et reprise

In [ ]:
EXPERIMENT_NAME = 'e015-aesthetic-backbone-reference-v1'
RESUME_RUN_NAME = None
PAYLOAD = 'https://ptag.io/t/e015'
PROMPT_LIMIT = None
PROMPTS = [
    {'id': 'p1_simple', 'seed': 1101, 'text': 'A single white lotus flower floating on a dark calm pond, elegant editorial photograph.'},
    {'id': 'p2_medium', 'seed': 2202, 'text': 'A Japanese garden with a red bridge, mossy stones and soft morning mist, detailed photography.'},
    {'id': 'p3_detailed', 'seed': 3303, 'text': 'An ornate botanical tapestry of white lilies, pale blue leaves and dark vines, intricate textile illustration.'},
    {'id': 'p4_complex', 'seed': 4404, 'text': 'A lively old European market square, café terraces, flowers, bicycles and a gothic cathedral, cinematic morning light.'},
]
ACTIVE_PROMPTS = PROMPTS[:PROMPT_LIMIT] if PROMPT_LIMIT else PROMPTS
NEGATIVE_PROMPT = 'text, watermark, letters, low quality, malformed'
CANVAS_SIZE = 768
QR_VERSION = 3
QR_MODULE_SIZE = 20
QR_ECC = 'M'
REQUIRE_ALL_MODELS = True
HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')

MODEL_SPECS = [
    {
        'name': 'sd15_cetus', 'kind': 'sd15',
        'repo': 'fp16-guy/Cetus-Mix_Whalefall_fp16_cleaned',
        'filename': 'cetusMix_Whalefall2_fp16.safetensors',
        'steps': 30, 'guidance': 7.5, 'offload_mode': 'none',
    },
    {
        'name': 'sdxl_base_1_0', 'kind': 'sdxl',
        'repo': 'stabilityai/stable-diffusion-xl-base-1.0',
        'steps': 30, 'guidance': 7.0, 'offload_mode': 'model_cpu',
    },
    {
        'name': 'flux_1_schnell', 'kind': 'flux',
        'repo': 'black-forest-labs/FLUX.1-schnell',
        'steps': 4, 'guidance': 0.0, 'offload_mode': 'sequential_cpu',
    },
]

if RESUME_RUN_NAME:
    RUN_DIR = Path('/data/notebook-runs') / RESUME_RUN_NAME
else:
    RUN_DIR = Path('/data/notebook-runs') / (
        datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + EXPERIMENT_NAME
    )
    RUN_DIR.mkdir(parents=True)
RESULTS_PATH = RUN_DIR / 'results.jsonl'
print('Sortie :', RUN_DIR)


## 2. Utilitaires mémoire, chargement séquentiel et métriques

In [ ]:
def release_pipeline(pipeline):
    if pipeline is not None:
        try:
            pipeline.to('cpu')
        except Exception:
            pass
        del pipeline
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def resolve_spec(spec, revision=None):
    resolved = dict(spec)
    resolved['revision'] = revision or model_info(
        spec['repo'], token=HF_TOKEN
    ).sha
    if spec['kind'] == 'sd15':
        resolved['local_model'] = hf_hub_download(
            repo_id=spec['repo'], filename=spec['filename'],
            revision=resolved['revision'], cache_dir='/cache/huggingface',
            token=HF_TOKEN,
        )
    return resolved


def check_model_access(spec, revision):
    filename = spec.get('filename', 'model_index.json')
    try:
        hf_hub_download(
            repo_id=spec['repo'], filename=filename, revision=revision,
            cache_dir='/cache/huggingface', token=HF_TOKEN,
        )
        return {'available': True, 'reason': None}
    except (GatedRepoError, HfHubHTTPError) as exc:
        status = getattr(getattr(exc, 'response', None), 'status_code', None)
        return {
            'available': False,
            'reason': f'{type(exc).__name__} (HTTP {status or "inconnu"})',
        }


def load_pipeline(spec):
    started = time.perf_counter()
    if spec['kind'] == 'sd15':
        pipeline = StableDiffusionPipeline.from_single_file(
            spec['local_model'], torch_dtype=torch.float16, cache_dir='/cache/huggingface',
            safety_checker=None, use_safetensors=True,
        )
        pipeline = pipeline.to('cuda')
    elif spec['kind'] == 'sdxl':
        pipeline = StableDiffusionXLPipeline.from_pretrained(
            spec['repo'], revision=spec['revision'],
            torch_dtype=torch.float16, variant='fp16',
            cache_dir='/cache/huggingface', use_safetensors=True, token=HF_TOKEN,
        )
        pipeline.enable_model_cpu_offload()
    elif spec['kind'] == 'flux':
        pipeline = FluxPipeline.from_pretrained(
            spec['repo'], revision=spec['revision'],
            torch_dtype=torch.bfloat16, cache_dir='/cache/huggingface',
            token=HF_TOKEN,
        )
        # FLUX 12B BF16 ne tient pas comme un bloc sur la RTX 20 Gio.
        # L'offload séquentiel charge seulement le sous-module en cours sur le GPU.
        pipeline.enable_sequential_cpu_offload()
        pipeline.vae.enable_slicing()
        pipeline.vae.enable_tiling()
    else:
        raise ValueError(spec['kind'])
    if hasattr(pipeline, 'enable_attention_slicing'):
        pipeline.enable_attention_slicing()
    return pipeline, time.perf_counter() - started


def generate_reference(pipeline, spec, case):
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    kwargs = {
        'prompt': case['text'], 'height': CANVAS_SIZE, 'width': CANVAS_SIZE,
        'num_inference_steps': spec['steps'],
        'generator': torch.Generator(
            device='cpu' if spec['kind'] == 'flux' else 'cuda'
        ).manual_seed(case['seed']),
    }
    if spec['kind'] == 'flux':
        kwargs.update({'guidance_scale': spec['guidance'], 'max_sequence_length': 256})
    else:
        kwargs.update({
            'negative_prompt': NEGATIVE_PROMPT, 'guidance_scale': spec['guidance'],
        })
    image = pipeline(**kwargs).images[0].convert('RGB')
    torch.cuda.synchronize()
    return image, time.perf_counter() - started, torch.cuda.max_memory_allocated() / 2**30


validator = QRValidator()
quality_scorer = CLIPQualityScorer(Path('/cache'), device='cpu')


def validate_exact(image):
    records = validator.validate(image, PAYLOAD)
    summary = summarize_validation_records(records)
    passed = sum(item.exact_payload_match for item in records)
    return {
        'passed': passed, 'total': len(records), 'pass_rate': passed / len(records),
        'strict_all': passed == len(records),
        'worst_decoder_pass_rate': summary['worst_decoder_pass_rate'],
        'worst_scenario_pass_rate': summary['worst_scenario_pass_rate'],
    }


def append_row(row):
    with RESULTS_PATH.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        stream.flush()


def completed():
    if not RESULTS_PATH.exists():
        return set()
    keys = {
        (row['model'], row['prompt_id'])
        for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()
        for row in [json.loads(line)]
    }
    for model_name, prompt_id in keys:
        output_dir = RUN_DIR / model_name / prompt_id
        required = [
            output_dir / 'reference.png', output_dir / 'adaptive-blueprint.png',
            output_dir / 'exact-mask-blueprint.png',
        ]
        if not all(path.exists() for path in required):
            raise RuntimeError(
                f'Résultat {model_name}/{prompt_id} indexé mais artefacts incomplets.'
            )
    return keys


## 3. Générer les douze références et leurs blueprints adaptatifs

In [ ]:
RESOLVED_PATH = RUN_DIR / 'resolved-model-revisions.json'
if RESOLVED_PATH.exists():
    saved_revisions = json.loads(RESOLVED_PATH.read_text(encoding='utf-8'))
else:
    saved_revisions = {
        spec['name']: model_info(spec['repo'], token=HF_TOKEN).sha
        for spec in MODEL_SPECS
    }
    RESOLVED_PATH.write_text(
        json.dumps(saved_revisions, indent=2), encoding='utf-8'
    )
access_report = {
    spec['name']: check_model_access(spec, saved_revisions[spec['name']])
    for spec in MODEL_SPECS
}
(RUN_DIR / 'model-access.json').write_text(
    json.dumps(access_report, indent=2), encoding='utf-8'
)
unavailable = [
    spec for spec in MODEL_SPECS
    if not access_report[spec['name']]['available']
]
if unavailable:
    details = ', '.join(
        f"{spec['repo']} [{access_report[spec['name']]['reason']}]"
        for spec in unavailable
    )
    message = (
        'Accès Hugging Face absent pour : ' + details + '. '
        'Pour FLUX.1-schnell, accepter d abord les conditions sur la page du modèle, '
        'créer le secret Kubernetes prooftag-huggingface avec une clé token, '
        'puis recréer le pod avec notebook-remote.ps1 -Reset.'
    )
    if REQUIRE_ALL_MODELS:
        raise RuntimeError(message)
    print('AVERTISSEMENT :', message)

resolved_specs = [
    resolve_spec(spec, saved_revisions[spec['name']])
    for spec in MODEL_SPECS
    if access_report[spec['name']]['available']
]

for spec in resolved_specs:
    pending = [case for case in ACTIVE_PROMPTS if (spec['name'], case['id']) not in completed()]
    if not pending:
        print('SKIP modèle complet :', spec['name'])
        continue
    pipeline, load_seconds = load_pipeline(spec)
    print(spec['name'], 'chargé en', round(load_seconds, 1), 's')
    for case in pending:
        output_dir = RUN_DIR / spec['name'] / case['id']
        output_dir.mkdir(parents=True, exist_ok=True)
        image, generation_seconds, peak_vram = generate_reference(
            pipeline, spec, case
        )
        image.save(output_dir / 'reference.png')

        exact = exact_mask_candidates(
            PAYLOAD, image, version=QR_VERSION, error_correction=QR_ECC,
            module_size=QR_MODULE_SIZE, canvas_size=CANVAS_SIZE,
        )[0].aligned
        adaptive_candidates = []
        for minimum_fraction in [0.22, 0.30, 0.38, 0.46, 0.55, 0.70, 0.85]:
            adaptive = build_adaptive_blueprint(
                image, exact, minimum_data_fraction=minimum_fraction
            )
            aligned_adaptive = AlignedQR(
                image=adaptive.image, core_matrix=exact.core_matrix.copy(),
                version=exact.version, error_correction=exact.error_correction,
                mask_pattern=exact.mask_pattern, module_size=exact.module_size,
                padding_px=exact.padding_px, canvas_size=exact.canvas_size, payload=exact.payload,
            )
            validation = validate_exact(adaptive.image)
            adaptive_candidates.append((validation, adaptive, aligned_adaptive, minimum_fraction))
        adaptive_candidates.sort(
            key=lambda item: (
                item[0]['strict_all'], item[0]['pass_rate'],
                -item[1].reference_cost, -item[1].grid_visibility,
            ),
            reverse=True,
        )
        validation, adaptive, aligned_adaptive, minimum_fraction = adaptive_candidates[0]
        adaptive.image.save(output_dir / 'adaptive-blueprint.png')
        exact.image.save(output_dir / 'exact-mask-blueprint.png')
        quality = asdict(quality_scorer.score(image, case['text']))
        row = {
            'model': spec['name'], 'kind': spec['kind'], 'model_id': spec['repo'],
            'resolved_revision': spec['revision'],
            'prompt_id': case['id'], 'prompt': case['text'], 'seed': case['seed'],
            'steps': spec['steps'], 'guidance': spec['guidance'],
            'offload_mode': spec['offload_mode'],
            'load_seconds': load_seconds, 'generation_seconds': generation_seconds,
            'peak_vram_gib': peak_vram, **quality,
            'adaptive_minimum_fraction': minimum_fraction,
            'adaptive_reference_cost': adaptive.reference_cost,
            'adaptive_grid_visibility': adaptive.grid_visibility,
            **{f'adaptive_{key}': value for key, value in validation.items()},
        }
        append_row(row)
        display(image.resize((384, 384)))
        display(adaptive.image.resize((384, 384)))
    release_pipeline(pipeline)


## 4. Comparaison appariée et décision pour la suite

In [ ]:
rows = [
    json.loads(line) for line in RESULTS_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
frame = pd.DataFrame(rows)
frame.to_csv(RUN_DIR / 'comparison.csv', index=False)
display(frame[[
    'model', 'prompt_id', 'generation_seconds', 'peak_vram_gib',
    'clip_aesthetic', 'clip_score', 'adaptive_pass_rate',
    'adaptive_reference_cost', 'adaptive_grid_visibility',
]])

aggregate = frame.groupby('model').agg(
    prompts=('prompt_id', 'nunique'),
    mean_seconds=('generation_seconds', 'mean'),
    max_vram_gib=('peak_vram_gib', 'max'),
    mean_aesthetic=('clip_aesthetic', 'mean'),
    worst_aesthetic=('clip_aesthetic', 'min'),
    mean_clip=('clip_score', 'mean'),
    adaptive_strict=('adaptive_strict_all', 'sum'),
    adaptive_worst_ssr=('adaptive_pass_rate', 'min'),
    adaptive_mean_cost=('adaptive_reference_cost', 'mean'),
).reset_index()
aggregate.to_csv(RUN_DIR / 'aggregate.csv', index=False)
display(aggregate.sort_values(
    ['adaptive_strict', 'adaptive_worst_ssr', 'mean_aesthetic', 'mean_clip'],
    ascending=False,
))

figure, axes = plt.subplots(1, 3, figsize=(18, 5))
for model_name, part in frame.groupby('model'):
    axes[0].scatter(part.clip_score, part.clip_aesthetic, label=model_name, s=70)
    axes[1].scatter(part.generation_seconds, part.clip_aesthetic, label=model_name, s=70)
    axes[2].scatter(part.adaptive_pass_rate, part.adaptive_reference_cost, label=model_name, s=70)
axes[0].set(xlabel='CLIPScore', ylabel='CLIP-aesthetic')
axes[1].set(xlabel='secondes', ylabel='CLIP-aesthetic')
axes[2].set(xlabel='SSR blueprint adaptatif', ylabel='coût visuel vs référence')
for axis in axes:
    axis.grid(alpha=0.25)
axes[0].legend()
figure.tight_layout()
figure.savefig(RUN_DIR / 'objectives.png', dpi=160)
display(figure)


## 5. Manifeste, limites et archive

In [ ]:
manifest = {
    'experiment': EXPERIMENT_NAME, 'models': MODEL_SPECS,
    'resolved_revisions': saved_revisions, 'prompts': ACTIVE_PROMPTS,
    'payload': PAYLOAD, 'canvas_size': CANVAS_SIZE,
    'selection_scope': 'aesthetic reference only; no claim about QR diffusion compatibility',
    'limits': [
        'Different recommended step budgets are an operational comparison, not equal compute.',
        'Same integer seed does not create identical latent noise across architectures.',
        'Adaptive blueprint validation does not predict a final ControlNet/SRPG image.',
        'FLUX and SDXL are not inserted into the SD1.5 DiffQRCoder pipeline here.',
    ],
}
(RUN_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(RUN_DIR), 'gztar', RUN_DIR.parent, RUN_DIR.name)
print('Archive :', archive)
